# Сравнение моделей

Соберу validation-предсказания TF-IDF, Transformer с нуля и DistilBERT в одну таблицу. Затем оценю выбранную модель на официальном test и посмотрю, какие темы она путает.

Это первое сравнение. Отдельный эксперимент с передачей обращений оператору и замером скорости находится в [ноутбуке 06](06_routing_and_latency.ipynb).

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DATA_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")
PREDICTIONS_DIR = REPORTS_DIR / "predictions"

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Device:", device)
print("Test:", test_df.shape)

sys.path.insert(0, str(Path("../src").resolve()))

Device: cuda
Test: (3080, 2)


## Validation: единая таблица результатов

In [2]:
prediction_files = {
    "tfidf": (PREDICTIONS_DIR / "tfidf_validation.csv"),
    "transformer_scratch": (PREDICTIONS_DIR / "transformer_scratch_validation.csv"),
    "distilbert": (PREDICTIONS_DIR / "distilbert_validation.csv"),
}

summary_files = {
    "tfidf": REPORTS_DIR / "tfidf_summary.json",
    "transformer_scratch": (REPORTS_DIR / "transformer_scratch_summary.json"),
    "distilbert": (REPORTS_DIR / "distilbert_summary.json"),
}

missing = [str(path) for path in prediction_files.values() if not path.exists()]

if missing:
    raise FileNotFoundError(
        f"Сначала запустите предыдущие ноутбуки. Не найдены файлы: {missing}"
    )

validation_tables = {}
comparison_rows = []

for model_name, path in prediction_files.items():
    predictions = pd.read_csv(path)
    validation_tables[model_name] = predictions

    summary = {}
    if summary_files[model_name].exists():
        with open(summary_files[model_name], encoding="utf-8") as file:
            summary = json.load(file)

    comparison_rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(
                predictions["category"], predictions["predicted_category"]
            ),
            "macro_f1": f1_score(
                predictions["category"],
                predictions["predicted_category"],
                average="macro",
            ),
            "errors": (
                predictions["category"] != predictions["predicted_category"]
            ).sum(),
            "training_seconds": summary.get("training_seconds", np.nan),
            "trainable_parameters": summary.get("trainable_parameters", np.nan),
        }
    )

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(comparison_df)

,model,accuracy,macro_f1,errors,training_seconds,trainable_parameters
0,distilbert,0.8915,0.881844,217,100.625201,67012685.0
1,tfidf,0.8650,0.861327,270,0.580442,NaN
2,transformer_scratch,0.8235,0.824073,353,19.481203,456013.0


## Какие интенты путает лучшая модель

In [3]:
best_model_name = comparison_df.loc[0, "model"]
best_validation = validation_tables[best_model_name]

best_errors = best_validation.query("category != predicted_category")

confusion_pairs = (
    best_errors.groupby(["category", "predicted_category"])
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
)

print("Лучшая модель:", best_model_name)
display(confusion_pairs.head(15))

error_sample = best_errors.sample(n=min(25, len(best_errors)), random_state=42)

display(error_sample[["text", "category", "predicted_category", "confidence"]])

error_sample.to_csv(REPORTS_DIR / "error_analysis_candidates.csv", index=False)

Лучшая модель: distilbert


,category,predicted_category,count
0,why_verify_identity,verify_my_identity,14
1,transfer_not_received_by_recipient,balance_not_updated_after_bank_transfer,7
2,virtual_card_not_working,getting_virtual_card,6
3,direct_debit_payment_not_recognised,card_payment_not_recognised,5
4,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate,5
5,wrong_amount_of_cash_received,cash_withdrawal_charge,4
6,transfer_not_received_by_recipient,transfer_timing,4
7,fiat_currency_support,exchange_via_app,3
8,declined_transfer,declined_card_payment,3
9,pending_top_up,top_up_failed,3


,text,category,predicted_category,confidence
1945,How do I activate and link my new card?,card_linking,activate_my_card,0.914201
1977,I made a transfer recently but it has not chan...,pending_transfer,balance_not_updated_after_bank_transfer,0.842162
1279,I did all that was asked of me and I am rather...,Refund_not_showing_up,reverted_card_payment?,0.495114
1703,I cant transfer money into my account,beneficiary_not_allowed,transfer_into_account,0.317320
137,What proof do you need for my identification?,unable_to_verify_identity,verify_my_identity,0.462307
1086,What is my money worth in other countries?,exchange_rate,country_support,0.243343
1731,What was the issue with my topping up?,top_up_failed,top_up_reverted,0.370686
684,What happens if a merchant doesn't accept you...,reverted_card_payment?,declined_card_payment,0.374230
1946,How many payments can I make using a virtual c...,disposable_card_limits,getting_virtual_card,0.463538
1293,Is there a way for you to find out why my acco...,direct_debit_payment_not_recognised,reverted_card_payment?,0.526944


## Загрузка выбранной модели

Для оценки использую сохранённые веса и ту же обработку текста, что и при обучении.

In [4]:
from banking_intent.scratch import IntentTransformer, prepare_text


def predict_tfidf(texts: list[str]) -> tuple[list[str], list[float]]:
    model = joblib.load(MODELS_DIR / "tfidf_logreg.joblib")
    predictions = model.predict(texts).tolist()
    confidence = model.predict_proba(texts).max(axis=1).tolist()
    return (predictions, confidence)


def prepare_scratch_batch(texts, token_to_id, max_length):
    encoded = [prepare_text(text, token_to_id, max_length) for text in texts]
    input_ids, attention_masks = zip(*encoded)
    return (
        torch.tensor(input_ids, dtype=torch.long),
        torch.tensor(attention_masks, dtype=torch.long),
    )


def predict_scratch(
    texts: list[str], batch_size: int = 64
) -> tuple[list[str], list[float]]:
    with open(MODELS_DIR / "transformer_metadata.json", encoding="utf-8") as file:
        metadata = json.load(file)
    label_to_id = metadata["label_to_id"]
    id_to_label = {index: label for label, index in label_to_id.items()}
    model = IntentTransformer(
        vocab_size=len(metadata["token_to_id"]),
        num_classes=len(label_to_id),
        max_length=metadata["max_length"],
        **metadata["model_config"],
    ).to(device)
    weights_path = MODELS_DIR / "transformer_from_scratch.pt"
    try:
        state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    except TypeError:
        state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    predictions = []
    confidence = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start : start + batch_size]
            input_ids, attention_mask = prepare_scratch_batch(
                batch_texts, metadata["token_to_id"], metadata["max_length"]
            )
            logits = model(input_ids.to(device), attention_mask.to(device))
            probabilities = torch.softmax(logits, dim=1)
            batch_confidence, batch_ids = probabilities.max(dim=1)
            predictions.extend(
                (id_to_label[index] for index in batch_ids.cpu().tolist())
            )
            confidence.extend(batch_confidence.cpu().tolist())
    return (predictions, confidence)


def predict_distilbert(
    texts: list[str], batch_size: int = 32
) -> tuple[list[str], list[float]]:
    model_dir = MODELS_DIR / "distilbert_banking77"
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    predictions = []
    confidence = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start : start + batch_size]
            encoded = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=64,
                return_tensors="pt",
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            logits = model(**encoded).logits
            probabilities = torch.softmax(logits, dim=1)
            batch_confidence, batch_ids = probabilities.max(dim=1)
            predictions.extend(
                (model.config.id2label[index] for index in batch_ids.cpu().tolist())
            )
            confidence.extend(batch_confidence.cpu().tolist())
    return (predictions, confidence)

## Результат на test

In [5]:
predictors = {
    "tfidf": predict_tfidf,
    "transformer_scratch": predict_scratch,
    "distilbert": predict_distilbert,
}

final_predictor = predictors[best_model_name]

test_predictions, test_confidence = final_predictor(test_df["text"].tolist())

test_accuracy = accuracy_score(test_df["category"], test_predictions)
test_macro_f1 = f1_score(test_df["category"], test_predictions, average="macro")

print("Финальная модель:", best_model_name)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro F1: {test_macro_f1:.4f}")

final_test_predictions = test_df.assign(
    predicted_category=test_predictions, confidence=test_confidence
)

final_test_predictions.to_csv(
    PREDICTIONS_DIR / "final_test_predictions.csv", index=False
)

final_summary = {
    "selected_model": best_model_name,
    "validation_macro_f1": float(comparison_df.loc[0, "macro_f1"]),
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
}

with open(REPORTS_DIR / "final_metrics.json", "w", encoding="utf-8") as file:
    json.dump(final_summary, file, ensure_ascii=False, indent=2)

Финальная модель: distilbert
Test accuracy: 0.8893
Test macro F1: 0.8864
